# Connect to RabbitMQ

In [13]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent.parent))

from communication import protocol
from communication.rabbitmq import Rabbitmq

# Initialize RabbitMQ connection (adjust parameters as needed)
try:
    rmq = Rabbitmq(
        ip="localhost",
        port=5672,
        username="ur3e",
        password="ur3e",
        vhost="/",
        exchange="UR3E_AMQP",
        type="topic",
    )
    rmq.connect_to_server()
    print("✓ Connected to RabbitMQ successfully")
except Exception as e:
    print(f"✗ Failed to connect to RabbitMQ: {e}")
    print("\nMake sure RabbitMQ is running. You can start it with:")
    print("  python -m startup.start_docker_rabbitmq")

def send_control_message(rmq, msg):
    """Send a control message to the UR3e Mockup via RabbitMQ."""
    try:
        rmq.send_message(
            routing_key=protocol.ROUTING_KEY_CTRL,
            message=msg
        )
        print(f"✓ Control message: {msg} sent successfully")
    except Exception as e:
        print(f"✗ Failed to send control message: {e}")

✓ Connected to RabbitMQ successfully


# Random movement generator

The block below will create random movements. Every time the arm has finished a movement press ctrl+enter to generate a new movement.

In [14]:
import numpy as np
import time

# Construct control message for loading a program
position_np = (np.random.rand((6)) - 0.5) * 0.5*np.pi # Generate 6 random joint positions
position = position_np.tolist()
vel = 60 # deg/s
acc = 80 # deg/s²

msg = {
    protocol.CtrlMsgKeys.TYPE: protocol.CtrlMsgFields.LOAD_PROGRAM,
    protocol.CtrlMsgKeys.JOINT_POSITIONS: [position],
    protocol.CtrlMsgKeys.MAX_VELOCITY: vel,
    protocol.CtrlMsgKeys.ACCELERATION: acc,
}

send_control_message(rmq, msg)

# send control message for starting program
msg_start = {
    protocol.CtrlMsgKeys.TYPE: protocol.CtrlMsgFields.PLAY,
}

send_control_message(rmq, msg_start)


✓ Control message: {'type': 'load_program', 'joint_positions': [[0.2743583035568329, -0.6177506302205241, -0.5366519378144744, -0.7703621398469005, 0.7088500454780211, -0.2737824251861126]], 'max_velocity': 60, 'acceleration': 80} sent successfully
✓ Control message: {'type': 'play'} sent successfully


# Automatic movement generator

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output, display
from influxdb_client import InfluxDBClient

# Configuration (working for Hugo)
URL = "http://localhost:8086"
TOKEN = "SECRET_AF"
ORG = "ur3e"
BUCKET = "ur3e"
threshold = 0.01 

client = InfluxDBClient(url=URL, token=TOKEN, org=ORG)
query_api = client.query_api()

def get_max_robot_velocity(api, bkt):
    query = f'''
    from(bucket: "{bkt}")
      |> range(start: -3s)
      |> filter(fn: (r) => r["_measurement"] == "sensor_data")
      |> filter(fn: (r) => r["_field"] =~ /qd_actual_joint_[0-5]/)
      |> last()
    '''
    try:
        result = api.query(query)
        velocities = [abs(record.get_value()) for table in result for record in table.records]
        return max(velocities) if velocities else None
    except Exception:
        return None

# I used this function to plot the robot joint positions to ensure that the robot was moving accordingly.
def update_jupyter_plot(api, bkt):
    """Refresh the plot"""
    query = f'''
    from(bucket: "{bkt}")
      |> range(start: -2m)
      |> filter(fn: (r) => r["_measurement"] == "sensor_data")
      |> filter(fn: (r) => r["_field"] =~ /q_actual_joint_[0-5]/)
      |> pivot(rowKey:["_time"], columnKey: ["_field"], valueColumn: "_value")
    '''
    try:
        df = api.query_data_frame(query)
        
        if df is not None and not df.empty:
            clear_output(wait=True) # Clear previous plot
            
            plt.figure(figsize=(12, 5))
            for i in range(6):
                field = f"q_actual_joint_{i}"
                if field in df.columns:
                    plt.plot(df['_time'], df[field], label=f"Joint {i}") # Actualize plot with current positions
            
            plt.title(f"Robot Trajectory (Last 2 min) - Updated at {time.strftime('%H:%M:%S')}")
            plt.ylabel("Position (rad)")
            plt.xlabel("Time")
            plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
            plt.grid(True, alpha=0.3)
            plt.xticks(rotation=25)
            plt.show()
        else:
            print("No data.")
            
    except Exception as e:
        print(f"Plotting error: {e}")

# Random Movement Generation
print("Starting Random Motion...")

while True:
    position = (np.random.uniform(-1, 1, 6) * np.pi).tolist() # Generate random joint positions within [-pi, pi]
    msg_load = {
        protocol.CtrlMsgKeys.TYPE: protocol.CtrlMsgFields.LOAD_PROGRAM,
        protocol.CtrlMsgKeys.JOINT_POSITIONS: [position],
        protocol.CtrlMsgKeys.MAX_VELOCITY: 60,
        protocol.CtrlMsgKeys.ACCELERATION: 80,
    }
    send_control_message(rmq, msg_load)
    send_control_message(rmq, { protocol.CtrlMsgKeys.TYPE: protocol.CtrlMsgFields.PLAY })

    # Monitor stationarity
    time.sleep(1.5)
    moving = True
    while moving:
        v_max = get_max_robot_velocity(query_api, BUCKET)
        
        if v_max is not None and v_max < threshold:
            #update_jupyter_plot(query_api, BUCKET) # Uncomment if you want to see the plot updating in real-time
            moving = False
        else:
            time.sleep(0.5)

Starting Random Motion...
✓ Control message: {'type': 'load_program', 'joint_positions': [[0.1521091694015269, -2.615730425933015, -1.99321595053588, 0.22422037326968314, 1.4945480973148135, 0.01083011627503002]], 'max_velocity': 60, 'acceleration': 80} sent successfully
✓ Control message: {'type': 'play'} sent successfully
✓ Control message: {'type': 'load_program', 'joint_positions': [[-1.5316635088952504, 1.9412885864504832, -2.795054426585227, -1.1959920729696267, -0.5535979496277732, 1.622606654362013]], 'max_velocity': 60, 'acceleration': 80} sent successfully
✓ Control message: {'type': 'play'} sent successfully
✓ Control message: {'type': 'load_program', 'joint_positions': [[1.8617429543089703, -0.1263648181799538, 1.5051619169184005, 0.45423903430483203, -2.616195627237504, 1.4787637868320205]], 'max_velocity': 60, 'acceleration': 80} sent successfully
✓ Control message: {'type': 'play'} sent successfully
✓ Control message: {'type': 'load_program', 'joint_positions': [[1.50871

KeyboardInterrupt: 

# Automatic movement generator (Old version)

In [3]:
import time
import numpy as np

while True:
    # Construct control message for loading a program
    position_np = (np.random.normal(0, 1, (6)) - 0.5) * 2*np.pi # Generate 6 random joint positions [-pi, pi]
    position = position_np.tolist()

    max_vel_bounds = (0, 360) # deg/s
    max_acc_bounds = (0, 180) # deg/s²      *Assumed

    def random_from_range(range: tuple):
        return np.random.rand() * (range[1] - range[0]) + range[0]

    vel = random_from_range(max_vel_bounds) # deg/s
    acc = random_from_range(max_acc_bounds) # deg/s²

    msg = {
        protocol.CtrlMsgKeys.TYPE: protocol.CtrlMsgFields.LOAD_PROGRAM,
        protocol.CtrlMsgKeys.JOINT_POSITIONS: [position],
        protocol.CtrlMsgKeys.MAX_VELOCITY: vel,
        protocol.CtrlMsgKeys.ACCELERATION: acc,
    }

    send_control_message(rmq, msg)

    # send control message for starting program
    msg_start = {
        protocol.CtrlMsgKeys.TYPE: protocol.CtrlMsgFields.PLAY,
    }

    send_control_message(rmq, msg_start)

    time.sleep(10)

    

# TODO:
# - Should generate random movement CTRL messages
# - The movements should be uniformly distributed
# - A new message should only be sent after the previous motion is done (Wait for stationarity)
  # - Lowpass filter over past N states, if velocity approx 0, send new control message
# - The messages should contain random joint positions, max velocity and acceleration


✓ Control message: {'type': 'load_program', 'joint_positions': [[-10.565164136099277, -2.688928031661978, -13.646239536053237, -1.6177143164352448, 3.609858988603059, -1.8693078259932492]], 'max_velocity': 89.60408885876723, 'acceleration': 1.3239736943526204} sent successfully
✓ Control message: {'type': 'play'} sent successfully
✓ Control message: {'type': 'load_program', 'joint_positions': [[-5.852077268228946, -1.2285838256752053, -0.5485638446594532, -7.6484801710080115, -6.677060402192959, -2.2950328461381106]], 'max_velocity': 303.18812364573483, 'acceleration': 108.52721792338177} sent successfully
✓ Control message: {'type': 'play'} sent successfully
✓ Control message: {'type': 'load_program', 'joint_positions': [[3.7061420548487773, -12.683790687207706, -11.71924072700502, 4.068637213206102, -15.244978275424733, -9.146595574843536]], 'max_velocity': 185.14253422059815, 'acceleration': 15.26925815618516} sent successfully
✓ Control message: {'type': 'play'} sent successfully
✓

KeyboardInterrupt: 